In [1]:
pip install -r ../requirements.txt --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
import re
import time

import numpy as np
import pandas as pd
import requests
import xarray as xr
from bs4 import BeautifulSoup

from ocean_features import add_time_features, pier_daily_features as _pier_daily_features

In [4]:
from dotenv import load_dotenv
load_dotenv(dotenv_path='../.env', override=True)

True

In [6]:


class OceanDataGenerator:
    """
    Aggregates oceanographic and meteorological data to build a dataset for 
    underwater visibility prediction.

    Data sources include CDIP (buoys), NOAA (wind/tide), OpenWeatherMap (rain),
    and parsed dive reports.

    Attributes:
        buoy_id (str): CDIP buoy station ID.
        noaa_wind_station (str): NOAA station ID for wind and water level.
        owm_api_key (str): OpenWeatherMap API key.
        lat (float): Latitude for weather data.
        lon (float): Longitude for weather data.
    """

    def __init__(self):
        self.buoy_id = "201" # SCRIPPS NEARSHORE, CA
        self.buoy_features = [
            'waveHs', 'waveTp', 'waveTa', 'waveDp', 
            'wavePeakPSD', 'sstSeaSurfaceTemperature'
        ]
        self.noaa_wind_station = "9410230" # Scripps Pier
        self.ocean_data_path = os.getenv("OCEAN_DATA_PATH")
        
        # Securely load API key from .env file
        self.owm_api_key = os.getenv("OWM_API_KEY")
        
        self.lat, self.lon = 32.8328, -117.2713
        self.headers = {'User-Agent': 'Mozilla/5.0'}
        
        # Incremental pull config: each source is cached as a daily parquet and
        # later runs only fetch days after the last cached date.
        self.cache_dir = "../data/raw"
        self.overlap_days = 3  # re-fetch recent days; the latest cached day is often partial

        # SCCOOS Scripps Pier in-water sensors (ERDDAP), cached as raw QC-filtered readings
        self.pier_url = "https://erddap.cencoos.org/erddap/tabledap/scripps-pier-automated-shore-sta-1.csv"
        self.pier_vars = {
            'turbidity': 'sea_water_turbidity_eco',
            'chl': 'mass_concentration_of_chlorophyll_in_sea_water_eco',
            'cdom': 'fluorescent_dissolved_organic_matter_eco',
        }
        # Local hour the model is run at. Daily pier features only see readings before this
        # hour, so training matches what is available at inference (e.g. an 8 pm run).
        self.pier_asof_hour = 20

    def _apply_circular_transform(self, df, column):
        """
        Decomposes a directional column (degrees) into sine and cosine components.
        
        This prevents machine learning models from misinterpreting the discontinuity 
        between 359 and 0 degrees.

        Args:
            df (pd.DataFrame): The dataframe containing the column.
            column (str): The name of the column containing degrees (0-360).

        Returns:
            pd.DataFrame: The dataframe with added sine and cosine columns.
        """
        rads = np.deg2rad(df[column])
        df[f'{column}_sine'] = np.sin(rads)
        df[f'{column}_cos'] = np.cos(rads)
        return df

    def _read_cache(self, name):
        """Returns the cached daily frame for a source, or an empty frame if none exists."""
        path = os.path.join(self.cache_dir, f"{name}.parquet")
        if os.path.exists(path):
            try:
                return pd.read_parquet(path, engine="fastparquet")
            except Exception as e:
                print(f"Ignoring unreadable cache {path}: {e}")
        return pd.DataFrame()

    def _fetch_start(self, name, days, full_refresh=False):
        """
        Decides where a pull should begin.

        Returns:
            tuple: (start Timestamp, cached DataFrame). Start is the requested window start
            on a cold or insufficient cache, otherwise the last cached day minus overlap_days.
        """
        requested = pd.Timestamp.now().normalize() - pd.Timedelta(days=days)
        cached = pd.DataFrame() if full_refresh else self._read_cache(name)
        if cached.empty or cached.index.min() > requested + pd.Timedelta(days=2):
            return requested, cached
        start = max(requested, cached.index.max().normalize() - pd.Timedelta(days=self.overlap_days))
        print(f"{name}: cached through {cached.index.max().date()}, fetching from {start.date()}")
        return start, cached

    def _merge_cache(self, name, cached, new, start=None):
        """
        Appends fresh rows to the cache (fresh rows win on overlap), persists it,
        and returns the combined frame trimmed to `start` if given.
        """
        frames = [f for f in (cached, new) if not f.empty]
        if not frames: return pd.DataFrame()
        combined = pd.concat(frames)
        combined = combined[~combined.index.duplicated(keep='last')].sort_index()
        os.makedirs(self.cache_dir, exist_ok=True)
        combined.to_parquet(os.path.join(self.cache_dir, f"{name}.parquet"), engine="fastparquet")
        return combined if start is None else combined[combined.index >= start]

    def fetch_buoy_data(self, days=650, full_refresh=False):
        """
        Fetches and aggregates wave and sea surface temperature data from CDIP THREDDS servers.

        Handles both historic (archived) and realtime netCDF files.

        Args:
            days (int): Number of days of history to fetch.

        Returns:
            pd.DataFrame: Daily aggregated wave and temperature data.
        """
        base_url = "http://thredds.cdip.ucsd.edu/thredds/dodsC/cdip"
        urls = [
            f"{base_url}/archive/{self.buoy_id}p1/{self.buoy_id}p1_historic.nc",
            f"{base_url}/realtime/{self.buoy_id}p1_rt.nc"
        ]
        
        start, cached = self._fetch_start('buoy', days, full_refresh)
        start_date = start.strftime('%Y-%m-%d')
        all_wave_aggs, all_sst_aggs = [], []

        for url in urls:
            try:
                ds = xr.open_dataset(url, engine='pydap')
                ds_subset = ds[self.buoy_features].sel(waveTime=slice(start_date, None))
                
                df_wave = ds_subset.drop_dims('sstTime').to_dataframe()
                df_wave = self._apply_circular_transform(df_wave, 'waveDp')
                
                # Feature engineering before daily aggregation
                df_wave['wave_steepness'] = df_wave['waveHs'] / df_wave['waveTp']
                df_wave['swell_energy'] = (df_wave['waveHs']**2) * df_wave['waveTp']
                
                wave_agg = df_wave.resample('D').agg({
                    'waveHs': ['max', 'mean'], 
                    'waveTp': 'mean',
                    'wave_steepness': 'mean',
                    'swell_energy': 'mean',
                    'waveDp_sine': 'mean', 
                    'waveDp_cos': 'mean', 
                    'wavePeakPSD': 'max'
                })
                # Flatten MultiIndex columns (e.g., ('waveHs', 'max') -> 'waveHs_max')
                wave_agg.columns = ['_'.join(col).strip() if isinstance(col, tuple) else col for col in wave_agg.columns.values]
                all_wave_aggs.append(wave_agg)

                try:
                    df_sst = ds_subset.drop_dims('waveTime').to_dataframe()
                    all_sst_aggs.append(df_sst.resample('D').agg({'sstSeaSurfaceTemperature': 'mean'}))
                except: pass
            except Exception as e:
                print(f"Skipping {url}: {e}")

        if not all_wave_aggs or not all_sst_aggs:
            return cached  # every source failed; fall back to what we already have
        final_wave = pd.concat(all_wave_aggs).sort_index()
        final_wave = final_wave[~final_wave.index.duplicated(keep='last')]
        final_sst = pd.concat(all_sst_aggs).sort_index()
        final_sst = final_sst[~final_sst.index.duplicated(keep='last')]

        fresh = final_wave.join(final_sst, how='inner')
        fresh.index = pd.to_datetime(fresh.index).normalize()
        return self._merge_cache('buoy', cached, fresh, start=pd.Timestamp.now().normalize() - pd.Timedelta(days=days))

    def fetch_wind_data(self, days=650, full_refresh=False):
        """
        Fetches wind speed, gust, and direction from NOAA API in 30-day chunks.
        
        Args:
            days (int): Number of days of history to fetch.
            
        Returns:
            pd.DataFrame: Daily aggregated wind vectors and speeds.
        """
        end_date = pd.Timestamp.now().normalize()
        start_date, cached = self._fetch_start('wind', days, full_refresh)
        all_chunks, curr = [], start_date
        
        while curr < end_date:
            curr_end = min(curr + pd.Timedelta(days=30), end_date)
            url = (f"https://api.tidesandcurrents.noaa.gov/api/prod/datagetter?"
                   f"begin_date={curr.strftime('%Y%m%d')}&end_date={curr_end.strftime('%Y%m%d')}&"
                   f"station={self.noaa_wind_station}&product=wind&units=metric&time_zone=lst_ldt&format=json")
            try:
                res = requests.get(url, headers=self.headers, timeout=15).json()
                if 'data' in res: all_chunks.append(pd.DataFrame(res['data']))
            except: pass
            # Advance slightly to avoid overlap or gap issues in API request
            curr = curr_end + pd.Timedelta(minutes=6)

        if not all_chunks: return cached
        df = pd.concat(all_chunks).drop_duplicates('t')
        df['t'] = pd.to_datetime(df['t'])
        df.set_index('t', inplace=True)
        for col in ['s', 'g', 'd']: df[col] = pd.to_numeric(df[col], errors='coerce')
        
        # Vectorize wind direction to calculate correct daily mean
        df['wind_x'], df['wind_y'] = np.cos(np.radians(df['d'])), np.sin(np.radians(df['d']))
        daily = df.resample('D').agg({'s': 'mean', 'g': 'max', 'wind_x': 'mean', 'wind_y': 'mean'}).rename(columns={'s': 'wind_speed', 'g': 'wind_gust'})
        daily['wind_dir_mean'] = np.degrees(np.arctan2(daily['wind_y'], daily['wind_x'])) % 360
        daily = daily.drop(columns=['wind_x', 'wind_y'])
        return self._merge_cache('wind', cached, daily, start=end_date - pd.Timedelta(days=days))
    
    def fetch_tide_data(self, days=650, full_refresh=False):
        """
        Fetches daily tidal max height and mean water level from NOAA.

        Args:
            days (int): Number of days of history to fetch.

        Returns:
            pd.DataFrame: Daily tide metrics.
        """
        end_date = pd.Timestamp.now().normalize()
        start_date, cached = self._fetch_start('tide', days, full_refresh)
        all_chunks, curr = [], start_date
        
        while curr < end_date:
            curr_end = min(curr + pd.Timedelta(days=30), end_date)
            url = (f"https://api.tidesandcurrents.noaa.gov/api/prod/datagetter?"
                   f"begin_date={curr.strftime('%Y%m%d')}&end_date={curr_end.strftime('%Y%m%d')}&"
                   f"station={self.noaa_wind_station}&product=water_level&datum=mllw&units=metric&time_zone=lst_ldt&format=json")
            try:
                res = requests.get(url, headers=self.headers, timeout=15).json()
                if 'data' in res: all_chunks.append(pd.DataFrame(res['data']))
            except: pass
            curr = curr_end + pd.Timedelta(minutes=6)

        if not all_chunks: return cached
        df = pd.concat(all_chunks)
        df['t'] = pd.to_datetime(df['t'])
        df.set_index('t', inplace=True)
        df['v'] = pd.to_numeric(df['v'], errors='coerce')
        
        daily = df.resample('D').agg({'v': ['max', 'mean']})
        daily.columns = ['tide_max', 'tide_mean']
        return self._merge_cache('tide', cached, daily, start=end_date - pd.Timedelta(days=days))

    def fetch_rain_data(self, days=365, full_refresh=False):
        """
        Fetches daily rain summary and calculates a 72-hour weighted accumulation.

        Args:
            days (int): Number of days of history to fetch.

        Returns:
            pd.DataFrame: Daily rain data including a weighted trailing metric.
        """
        rain_data = []
        end_date = pd.Timestamp.now().normalize()
        curr, cached = self._fetch_start('rain', days, full_refresh)
        
        while curr <= end_date:
            url = f"https://api.openweathermap.org/data/3.0/onecall/day_summary?lat={self.lat}&lon={self.lon}&date={curr.strftime('%Y-%m-%d')}&appid={self.owm_api_key}&units=metric"
            try:
                r = requests.get(url, timeout=15).json()
                # Skip error payloads so a bad response is never cached as a dry day
                if 'precipitation' in r:
                    rain_data.append({'time': curr, 'rain_mm': r['precipitation'].get('total', 0)})
            except: pass
            curr += pd.Timedelta(days=1)

        fresh = pd.DataFrame(rain_data, columns=['time', 'rain_mm']).set_index('time')
        fresh.index = pd.to_datetime(fresh.index)
        # Cache only the raw daily totals; the weighted metric is derived from the full series below
        df = self._merge_cache('rain', cached, fresh, start=end_date - pd.Timedelta(days=days))
        if df.empty: return df
        
        # Apply a decay factor: Today + (Yesterday * 0.6) + (2 Days Ago * 0.3)
        # This models runoff lag, where past rain still affects current water clarity.
        df['rain_72h_weighted_mm'] = (df['rain_mm'] + (df['rain_mm'].shift(1) * 0.6) + (df['rain_mm'].shift(2) * 0.3)).fillna(0)
        return df

    def fetch_pier_data(self, days=650, full_refresh=False):
        """
        Fetches turbidity, chlorophyll and CDOM readings (~every 4 min) from the SCCOOS
        Scripps Pier automated shore station via ERDDAP.

        Turbidity and chlorophyll keep only readings with QARTOD aggregate flag 1 (pass).
        SCCOOS does not QC CDOM (every reading is 'not evaluated'), so it gets a
        plausibility range instead. Bad readings become NaN.

        Args:
            days (int): Number of days of history to fetch.
            full_refresh (bool): Ignore the cache and re-pull.

        Returns:
            pd.DataFrame: Raw readings indexed by UTC timestamp, columns turbidity/chl/cdom.
        """
        import io
        end = pd.Timestamp.now().normalize() + pd.Timedelta(days=1)  # UTC, includes today
        start, cached = self._fetch_start('pier_raw', days, full_refresh)
        cols = ['time'] + [c for v in self.pier_vars.values() for c in (v, v + '_qc_agg')]

        frames, curr = [], start
        while curr < end:
            nxt = min(curr + pd.Timedelta(days=60), end)
            # '>' and '<' must be percent-encoded or the server returns HTTP 500
            url = (f"{self.pier_url}?{','.join(cols)}"
                   f"&time%3E={curr:%Y-%m-%dT%H:%M:%SZ}&time%3C{nxt:%Y-%m-%dT%H:%M:%SZ}")
            try:
                res = requests.get(url, headers=self.headers, timeout=180)
                if res.status_code == 200:
                    frames.append(pd.read_csv(io.StringIO(res.text), skiprows=[1]))  # row 2 is units
                else:
                    print(f"Pier pull {curr.date()} failed: HTTP {res.status_code}")
            except Exception as e:
                print(f"Pier pull {curr.date()} failed: {e}")
            curr = nxt

        if not frames: return cached
        df = pd.concat(frames)
        df['time'] = pd.to_datetime(df['time']).dt.tz_localize(None)
        df = df.set_index('time')

        out = pd.DataFrame(index=df.index)
        for name, v in self.pier_vars.items():
            val = pd.to_numeric(df[v], errors='coerce')
            if name == 'cdom':
                out[name] = val.where((val >= 0) & (val <= 100))
            else:
                out[name] = val.where(df[v + '_qc_agg'] == 1)
        out = out.dropna(how='all')
        return self._merge_cache('pier_raw', cached, out, start=end - pd.Timedelta(days=days + 1))

    def pier_daily_features(self, raw, min_readings=20):
        """Daily pier features as of pier_asof_hour; see ocean_features.pier_daily_features."""
        return _pier_daily_features(raw, self.pier_asof_hour, min_readings)

    def scrape_visibility_labels(self, total_pages=27, full_refresh=False):
        """
        Scrapes dive reports from a blog to extract visibility labels.

        Each report has a feet range in its header ("Vis: 10-15") and a rating graphic whose
        filename carries the shop's own category ("10-20_Good_<id>.png"). Some older posts use a
        generic placeholder image instead of the rating graphic; those have no usable category.

        Args:
            total_pages (int): Max pagination pages to scrape.
            full_refresh (bool): Ignore the cache and re-scrape every page.

        Returns:
            pd.DataFrame: Dates mapped to visibility_ft (lower bound of the header range) and
            category (0 Poor, 1 Fair, 2 Good, 3 Excellent; NaN when the post has no rating graphic).
        """
        base_url = "https://justgetwet.com/blogs/dive-reports-and-conditions?page="
        all_reports = []
        category_ids = {'poor': 0, 'fair': 1, 'good': 2, 'excellent': 3}
        cached = pd.DataFrame() if full_refresh else self._read_cache('labels')
        if 'category' not in cached.columns:
            cached = pd.DataFrame()  # cache predates the category column: re-scrape everything once
        known = set(cached.index)
        caught_up_pages = 0
        for page_num in range(1, total_pages + 1):
            page_dates = []
            try:
                res = requests.get(f"{base_url}{page_num}", headers=self.headers, timeout=10)
                soup = BeautifulSoup(res.text, 'html.parser')
                for art in soup.find_all('div', class_='article__grid-meta'):
                    date_tag, excerpt = art.find('time'), art.find('div', class_='article__excerpt')
                    if not date_tag or not excerpt: continue
                    
                    # Regex to find 'Viz: 10-15'' or 'Vis: 5m' patterns
                    viz_raw = re.search(r'(?:Viz|Vis|Visibility):\s*([\d\-\+m\s\']+)', excerpt.get_text(), re.IGNORECASE)
                    if viz_raw:
                        nums = re.findall(r'(\d+)', viz_raw.group(1).lower())
                        if not nums: continue
                        viz_ft = float(nums[0])
                        # Normalize meters to feet if 'm' is present
                        if 'm' in viz_raw.group(1).lower(): viz_ft *= 3.28
                        report_date = pd.to_datetime(date_tag.get_text().strip()).normalize()
                        page_dates.append(report_date)
                        # Rating graphic lives in the post's card; its filename holds the category
                        card = art.find_parent(lambda t: t.name in ('div', 'li', 'article') and t.find('img'))
                        imgs = ' '.join(' '.join(filter(None, [im.get('src'), im.get('data-src'), im.get('srcset')]))
                                        for im in card.find_all('img')) if card else ''
                        rating = re.search(r'\d+-\d+_(Poor|Fair|Good|Excellent)', imgs, re.IGNORECASE)
                        all_reports.append({'date': report_date, 'visibility_ft': viz_ft,
                                            'category': category_ids[rating.group(1).lower()] if rating else np.nan})
                time.sleep(0.5)
            except: pass
            # Pages run newest to oldest. Two straight pages of only cached reports means we've
            # caught up; requiring two guards against a single page that happens to be unchanged.
            if known and page_dates and all(d in known for d in page_dates):
                caught_up_pages += 1
                if caught_up_pages >= 2:
                    print(f"labels: pages {page_num - 1}-{page_num} already cached, stopping")
                    break
            else:
                caught_up_pages = 0
        fresh = pd.DataFrame(all_reports, columns=['date', 'visibility_ft', 'category']).drop_duplicates('date').set_index('date')
        return self._merge_cache('labels', cached, fresh)

    def run(self, days=650, scrape_pages=27, full_refresh=False, lookback_buffer_days=5):
        """
        Orchestrates the data pipeline: fetching, joining, and feature engineering.

        Args:
            days (int): Max days of history to fetch for features; capped at the earliest dive report.
            scrape_pages (int): Max blog pages to scrape for labels (stops early once caught up).
            full_refresh (bool): Ignore the cache in cache_dir and re-pull everything.
            lookback_buffer_days (int): Extra days kept before the earliest label when capping `days`.

        Returns:
            pd.DataFrame: The final cleaned dataset ready for training.
        """
        # Labels first: no point pulling feature days earlier than the oldest dive report.
        df_labels = self.scrape_visibility_labels(total_pages=scrape_pages, full_refresh=full_refresh)
        if not df_labels.empty:
            # Buffer covers the lag/diff/rain-shift features (up to 3 days) at the start of the window
            max_days = (pd.Timestamp.now().normalize() - df_labels.index.min()).days + lookback_buffer_days
            if days > max_days:
                print(f"Capping lookback from {days} to {max_days} days (earliest label: {df_labels.index.min().date()})")
                days = max_days

        df_buoy = self.fetch_buoy_data(days=days, full_refresh=full_refresh)
        df_wind = self.fetch_wind_data(days=days, full_refresh=full_refresh)
        df_rain = self.fetch_rain_data(days=days, full_refresh=full_refresh)
        df_tide = self.fetch_tide_data(days=days, full_refresh=full_refresh)
        df_pier = self.pier_daily_features(self.fetch_pier_data(days=days, full_refresh=full_refresh))
        self.df_pier = df_pier.copy()

        # Normalize indices to ensure clean joins
        for d in [df_buoy, df_wind, df_rain, df_tide, df_labels]:
            if not d.empty:
                d.index = pd.to_datetime(d.index).normalize().tz_localize(None)

        # Store intermediates for debugging/auditing
        self.df_buoy = df_buoy.copy() 
        self.df_wind = df_wind.copy()
        self.df_rain = df_rain.copy()
        self.df_tide = df_tide.copy()
        self.df_labels = df_labels.copy()
        
        audit_dict = {
            'df_buoy': self.df_buoy, 'df_wind': self.df_wind,
            'df_rain': self.df_rain, 'df_labels': self.df_labels,
            'df_tide': self.df_tide
        }
        for k, v in audit_dict.items():
            print(f"{k} min: {v.index.min()} {k} max: {v.index.max()}")
        
        daily = df_buoy.join(df_wind, how='inner').join(df_rain, how='inner').join(df_tide, how='inner')
        # Lags, SST changes, seasonality and swell trend are computed on the full daily series
        # BEFORE filtering to labeled days, so "1 day lag" is the previous calendar day (not the
        # previous labeled row, which can be several days back).
        daily = add_time_features(daily)
        final_df = daily.join(df_labels, how='inner')

        # Pier sensors: shift one calendar day so a label on day D only sees readings from D-1
        # (before pier_asof_hour). Left join, so days with no good sensor data stay NaN
        # (XGBoost handles NaN natively) instead of dropping the labeled row.
        if not df_pier.empty:
            final_df = final_df.join(df_pier.shift(1, freq='D').add_suffix('_lag1'), how='left')

        # Today's row has partial-day features, so only train on complete days
        final_df = final_df[final_df.index < pd.Timestamp.now().normalize()]
        required = [c for c in final_df.columns if not c.startswith('pier_') and c != 'category']
        self.data = final_df.dropna(subset=required)
        print(f"final_df Pre NA Drop {final_df.shape} Post NA Drop: {self.data.shape}")
        return self.data

    def update_observations(self, days=14):
        """
        Refreshes the raw caches (buoy, wind, tide, rain, pier) for recent days only, with no
        label scrape or training set rebuild. Run this before an evening forecast.

        Args:
            days (int): How far back to make sure the cache covers.
        """
        self.fetch_buoy_data(days=days)
        self.fetch_wind_data(days=days)
        self.fetch_rain_data(days=days)
        self.fetch_tide_data(days=days)
        self.fetch_pier_data(days=days)
        print(f"Caches in {self.cache_dir} refreshed.")

    def save_data(self, df, path, as_classification=False):
        """
        Saves the training frame to parquet with either a regression (feet) or classification (shop rating) target.

        Args:
            df (pd.DataFrame): The dataframe to save.
            path (str): Output filepath.
            as_classification (bool): If True, the target is the shop's own rating (0-3), and days
                                      whose post has no rating are dropped.
                                      If False, keeps visibility in feet as the regression target.
        """
        export_df = df.copy()
        if as_classification:
            export_df = export_df[export_df['category'].notna()]
            export_df['target'] = export_df['category'].astype(int)
            export_df.drop(columns=['visibility_ft', 'category'], inplace=True)
        else:
            export_df.drop(columns=['category'], errors='ignore', inplace=True)  # never a feature
            export_df.rename(columns={'visibility_ft': 'target'}, inplace=True)
            
        export_df.to_parquet(path, engine="fastparquet")
        print(f"Success. Parquet saved to {path} with {len(export_df)} records.")

In [ ]:
gen = OceanDataGenerator()
df = gen.run(1500, 40)  # days is capped at the earliest dive report
gen.save_data(df, path="../data/visibility_data_class.parquet", as_classification=True)
gen.save_data(df, path=gen.ocean_data_path, as_classification=False)

In [ ]:
# Evening run: refresh recent observations (seconds, incremental) before Model Inference's 8 pm forecast
gen = OceanDataGenerator()
gen.update_observations(days=14)